In [2]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import os

# Data analysis of the behaviors.parquet file

### Import

In [6]:
path = os.path.join('datas', 'ebnerd_demo', 'validation', 'behaviors.parquet') 
behaviors = pq.read_table(path).to_pandas()
print(behaviors)


       impression_id  article_id     impression_time  read_time  \
0             144772         NaN 2023-05-30 14:21:34       29.0   
1             144777         NaN 2023-05-30 14:22:11       10.0   
2             196487         NaN 2023-05-27 19:54:18       16.0   
3             196495         NaN 2023-05-27 19:53:48       25.0   
4             196496         NaN 2023-05-27 19:56:28       11.0   
...              ...         ...                 ...        ...   
25351      578776841         NaN 2023-05-26 20:46:25       63.0   
25352      578776846   9783317.0 2023-05-26 20:47:28       23.0   
25353      578777590   9780467.0 2023-05-26 20:43:52       21.0   
25354      578783544         NaN 2023-05-26 18:59:17       10.0   
25355      579246466         NaN 2023-05-26 08:08:12        5.0   

       scroll_percentage  device_type  \
0                    NaN            2   
1                    NaN            2   
2                    NaN            2   
3                    NaN       

### Creation of table with the user data

In [7]:
path = os.path.join('datas', 'ebnerd_demo', 'validation', 'behaviors.parquet')
data = pq.read_table(path).to_pandas()
users = data[['user_id', 'age', 'gender', 'postcode', 'is_subscriber', 'is_sso_user']]
print(users)


       user_id  age  gender  postcode  is_subscriber  is_sso_user
0        76658  NaN     NaN       NaN          False        False
1        76658  NaN     NaN       NaN          False        False
2       760446  NaN     NaN       NaN          False        False
3       760446  NaN     NaN       NaN          False        False
4       760446  NaN     NaN       NaN          False        False
...        ...  ...     ...       ...            ...          ...
25351  1167944  NaN     NaN       NaN          False        False
25352  1167944  NaN     NaN       NaN          False        False
25353  1171223  NaN     NaN       NaN          False        False
25354  1358150  NaN     NaN       NaN          False        False
25355  1514088  NaN     NaN       NaN          False        False

[25356 rows x 6 columns]


# analysis of the user data
Analysis on the amount of missing data and valid data and the percentage of valid data.


In [17]:
total_users = len(users)
missing_counts = users.isna().sum()

valid_counts = total_users - missing_counts

valid_percentages = (valid_counts / total_users) * 100

print("Missing values:\n", missing_counts)
print("\n Valid values:\n", valid_counts)
print("\n Percentage of valid values:\n", valid_percentages)

Missing values:
 user_id              0
age              24610
gender           23401
postcode         24979
is_subscriber        0
is_sso_user          0
dtype: int64

 Valid values:
 user_id          25356
age                746
gender            1955
postcode           377
is_subscriber    25356
is_sso_user      25356
dtype: int64

 Percentage of valid values:
 user_id          100.000000
age                2.942104
gender             7.710207
postcode           1.486828
is_subscriber    100.000000
is_sso_user      100.000000
dtype: float64


Missing values per attribute:
user_id              0
age              24610
gender           23401
postcode         24979
is_subscriber        0
is_sso_user          0

Valid values per attribute:
user_id          25356
age                746
gender            1955
postcode           377
is_subscriber    25356
is_sso_user      25356

Percentage of valid values per attribute:
user_id          100.000000
age                2.942104
gender             7.710207
postcode           1.486828
is_subscriber    100.000000
is_sso_user      100.000000

-> only a small percentage share their age, gender and postcode
Grouping the users by the age, gender or postcode would not be effective. Could as well lead to ethical problems given the gender bias on topics of articles. 

-> user_id, is_subscriber, is_sso_user are mandatory data


## Analysis based on if the user is subscriber


In [29]:
number_of_subscribers = (users['is_subscriber'] == 1).sum()

number_of_non_subscribers = (users['is_subscriber'] == 0).sum()

print("Number of subscribers: \n", number_of_subscribers)
print ("Number of non-subscribers: \n", number_of_non_subscribers)

optional_values = data[['user_id', 'age', 'gender', 'postcode', 'is_subscriber']] 
missing_values_by_subscriber = optional_values.groupby('is_subscriber')[['age', 'gender', 'postcode']].apply(lambda x: x.isna().sum())
print("Missing optional values by subscription status:\n", missing_values_by_subscriber)

valid_values_by_subscriber = optional_values.groupby('is_subscriber')[['age', 'gender', 'postcode']].apply(lambda x: x.notna().sum())
print("Available optional values by subscription status: \n", valid_values_by_subscriber)

subscribers = optional_values[optional_values['is_subscriber'] == 1]

total_subscribers = len(subscribers)

valid_counts_subscribers = subscribers[['age', 'gender', 'postcode']].notna().sum()

valid_percentages_subscribers = (valid_counts_subscribers / total_subscribers) * 100

print("Percentage of valid values for subscribers:\n", valid_percentages_subscribers)


Number of subscribers: 
 1595
Number of non-subscribers: 
 23761
Missing optional values by subscription status:
                  age  gender  postcode
is_subscriber                         
False          23451   22943     23576
True            1159     458      1403
Available optional values by subscription status: 
                age  gender  postcode
is_subscriber                       
False          310     818       185
True           436    1137       192
Percentage of valid values for subscribers:
 age         27.335423
gender      71.285266
postcode    12.037618
dtype: float64


Number of subscribers: 
 1595
Number of non-subscribers: 
 23761
 
Missing optional values by subscription status:
                  age  gender  postcode
is_subscriber                         
False          23451   22943     23576
True            1159     458      1403


Available optional values by subscription status: 
                age  gender  postcode
is_subscriber                       
False          310     818       185
True           436    1137       192

Percentage of valid values for subscribers:
age         27.335423
gender      71.285266
postcode    12.037618

-> not mandatory to answer age, gender and postcode when creating an user account 

-> only 12,04% provided their postcode, 27,34% their age and 71,29% their gender. 
Grouping by the gender seems to be the most promising idea.

## Analysis of the users grouped by their age

In [32]:
users_by_age = data[['user_id', 'age', 'is_subscriber']]

user_count_by_age = users_by_age.groupby('age').agg(
    total_users=('user_id', 'count'), 
    subscriber_count=('is_subscriber', lambda x: (x==1).sum()), 
    non_subscriber_count=('is_subscriber', lambda x: (x==0).sum())
).reset_index()

user_count_by_age = user_count_by_age.sort_values(by='age')
                                            
print(user_count_by_age)

    age  total_users  subscriber_count  non_subscriber_count
0   0.0            1                 0                     1
1  20.0           19                 1                    18
2  30.0          103                86                    17
3  40.0          132                69                    63
4  50.0          131               118                    13
5  60.0          182                96                    86
6  70.0          178                66                   112
